# Verifying Data Capture

After running an experiment, it's important to verify that data was captured correctly.

## Quick Verification Checklist

✅ Document file exists  
✅ File has non-zero size  
✅ Can load from DataBroker  
✅ Correct number of events  
✅ All expected columns present  
✅ Data values are reasonable  

## Setup and Run Test Experiment

In [ ]:
import sys
from pathlib import Path

parent_dir = Path.cwd().parent if 'jupyter_book_tutorial' in str(Path.cwd()) else Path.cwd()
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from bluesky import RunEngine
from ophyd import Signal
from bluesky.plans import scan
from bluesky_config.devices import MockDetector
from bluesky_config.callbacks import DocumentLogger
from databroker import Broker

# Setup with data capture
RE = RunEngine({})
doc_logger = DocumentLogger(str(parent_dir / 'data' / 'documents'))
RE.subscribe(doc_logger)
db = Broker.named('temp')
RE.subscribe(db.v1.insert)

# Devices
motor = Signal(name='motor', value=0)
detector = MockDetector(name='det')

# Run experiment
num_points = 15
uid = RE(scan([detector], motor, 0, 10, num_points))

print(f"✓ Experiment complete")
print(f"UID: {uid[0][:8]}")
print(f"Expected points: {num_points}")

## Verification 1: Check Document File

In [ ]:
# Check file exists
doc_file = parent_dir / 'data' / 'documents' / f"{uid[0]}_documents.jsonl"

if doc_file.exists():
    size = doc_file.stat().st_size
    print(f"✓ Document file exists")
    print(f"  Path: {doc_file}")
    print(f"  Size: {size:,} bytes")
    
    if size > 0:
        print(f"✓ File has data (non-zero size)")
    else:
        print(f"❌ File is empty!")
else:
    print(f"❌ Document file not found!")
    print(f"  Expected: {doc_file}")

## Verification 2: Load from DataBroker

In [ ]:
try:
    run = db[-1]
    print("✓ Can load from DataBroker")
    print(f"  UID: {run.metadata['start']['uid'][:8]}")
    print(f"  Plan: {run.metadata['start']['plan_name']}")
except Exception as e:
    print(f"❌ Failed to load from DataBroker: {e}")

## Verification 3: Check Event Count

In [ ]:
table = run.table()
actual_points = len(table)

print(f"Expected points: {num_points}")
print(f"Actual points: {actual_points}")

if actual_points == num_points:
    print(f"✓ Correct number of events captured")
else:
    print(f"❌ Event count mismatch!")
    print(f"  Missing {num_points - actual_points} events")

## Verification 4: Check Columns

In [ ]:
expected_columns = ['motor', 'det_value']
actual_columns = [col for col in table.columns if not col.endswith('_timestamp')]

print(f"Expected columns: {expected_columns}")
print(f"Actual columns: {actual_columns}")

missing = set(expected_columns) - set(actual_columns)
if not missing:
    print(f"✓ All expected columns present")
else:
    print(f"❌ Missing columns: {missing}")

## Verification 5: Data Quality Check

In [ ]:
import numpy as np

print("Data quality checks:")
print("="*50)

# Check for NaN values
has_nan = table[['motor', 'det_value']].isna().any().any()
if not has_nan:
    print("✓ No NaN values")
else:
    print("❌ Found NaN values!")

# Check motor range
motor_min, motor_max = table['motor'].min(), table['motor'].max()
print(f"\nMotor range: {motor_min:.2f} to {motor_max:.2f}")
if 0 <= motor_min and motor_max <= 10:
    print("✓ Motor values in expected range [0, 10]")

# Check detector values are reasonable
det_mean = table['det_value'].mean()
det_std = table['det_value'].std()
print(f"\nDetector stats:")
print(f"  Mean: {det_mean:.3f}")
print(f"  Std:  {det_std:.3f}")

if det_mean > 0:
    print("✓ Detector values are positive")

# Show data distribution
print(f"\nFirst 5 rows:")
print(table[['motor', 'det_value']].head())

## Verification 6: Metadata Check

In [ ]:
start = run.metadata['start']

print("Metadata verification:")
print("="*50)
print(f"Plan name: {start.get('plan_name', 'N/A')}")
print(f"Operator: {start.get('operator', 'N/A')}")
print(f"Scan ID: {start.get('scan_id', 'N/A')}")

if 'plan_args' in start:
    print(f"\nPlan arguments:")
    for key, val in start['plan_args'].items():
        print(f"  {key}: {val}")

# Check exit status
if 'stop' in run.metadata:
    exit_status = run.metadata['stop'].get('exit_status', 'unknown')
    print(f"\nExit status: {exit_status}")
    if exit_status == 'success':
        print("✓ Run completed successfully")
    else:
        print(f"❌ Run ended with status: {exit_status}")

## Complete Verification Summary

In [ ]:
print("\n" + "="*70)
print("VERIFICATION SUMMARY")
print("="*70)
print(f"\n✓ All verifications passed!")
print(f"\nData is ready for analysis.")
print(f"\nTo retrieve this run later:")
print(f"  from databroker import Broker")
print(f"  db = Broker.named('temp')")
print(f"  run = db['{uid[0][:8]}']")

## Troubleshooting Failed Verifications

### If document file doesn't exist:
- Check that `DocumentLogger` was subscribed to RunEngine
- Verify `data/documents/` directory exists
- Check file permissions

### If event count is wrong:
- Check if scan was interrupted
- Look for errors in console output
- Verify device triggering worked correctly

### If columns are missing:
- Check that all detectors were included in scan
- Verify device names match
- Check descriptor document for data_keys

### If data values look wrong:
- Check device configuration
- Verify units and scaling
- Look for connection issues with hardware

## Next Steps

Now that you can verify data was captured correctly, learn how to retrieve and analyze it:

→ **Chapter 5: [Retrieving and Analyzing Data](05_retrieval_analysis.ipynb)**